## Setup and Imports

In [1]:
import json
import os
import random
import numpy as np
import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset
from tqdm import tqdm
from collections import Counter

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Setup complete")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

C:\Users\super\Documents\UniPd\ATA\GutBrainIE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete
PyTorch version: 2.11.0.dev20260204+cu128
CUDA available: True


### Define Relation Labels and Configuration

This section defines the legal entity categories and relation predicates allowed in the dataset.
These labels correspond to the ontology used in the GutBrainIE relation extraction task.

Defining them early ensures that:
- only valid entity types are processed
- only valid relations are used during training and evaluation

In [2]:
import re

LEGAL_ENTITY_LABELS = {
    "anatomical location","animal","bacteria","biomedical technique","chemical","DDF",
    "dietary supplement","drug","food","gene","human","microbiome","statistical technique"
}
LEGAL_RELATION_LABELS = {
    "administered","affect","change abundance","change effect","change expression","compared to",
    "impact","influence","interact","is a","is linked to","located in","part of","produced by",
    "strike","target","used by"
}

def norm_ent(label: str) -> str:
    if label is None:
        return ""
    lab = str(label).strip()
    if lab.lower() == "ddf":
        return "DDF"
    return lab

def norm_span(s: str) -> str:
    # consigliato per ridurre mismatch banali sugli span
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

### Define Legal Entity and Relation Labels

This cell defines sets containing the allowed entity labels and relation labels.

These sets are used to:
- validate dataset annotations
- filter invalid relations
- ensure the training data respects the schema defined by the task

In [3]:
# Define legal relation predicates
RELATION_LABELS = [
    "no relation",  # For negative samples
    "administered",
    "affect",
    "change abundance",
    "change effect",
    "change expression",
    "compared to",
    "impact",
    "influence",
    "interact",
    "is a",
    "is linked to",
    "located in",
    "part of",
    "produced by",
    "strike",
    "target",
    "used by"
]

label2id = {label: idx for idx, label in enumerate(RELATION_LABELS)}
id2label = {idx: label for idx, label in enumerate(RELATION_LABELS)}

print(f"Total relation labels: {len(RELATION_LABELS)}")
print(f"Labels: {RELATION_LABELS}")

# Define legal entity type relations (subject_label, predicate, object_label)
# Order matters: relation is from subject to object
LEGAL_RELATIONS = [
    ("DDF", "affect", "DDF"),
    ("microbiome", "is linked to", "DDF"),
    ("DDF", "target", "human"),
    ("drug", "change effect", "DDF"),
    ("DDF", "is a", "DDF"),
    ("microbiome", "located in", "human"),
    ("chemical", "influence", "DDF"),
    ("dietary supplement", "influence", "DDF"),
    ("DDF", "target", "animal"),
    ("chemical", "impact", "microbiome"),
    ("anatomical location", "located in", "animal"),
    ("microbiome", "located in", "animal"),
    ("chemical", "located in", "anatomical location"),
    ("bacteria", "part of", "microbiome"),
    ("DDF", "strike", "anatomical location"),
    ("drug", "administered", "animal"),
    ("bacteria", "influence", "DDF"),
    ("drug", "impact", "microbiome"),
    ("DDF", "change abundance", "microbiome"),
    ("microbiome", "located in", "anatomical location"),
    ("microbiome", "used by", "biomedical technique"),
    ("chemical", "produced by", "microbiome"),
    ("dietary supplement", "impact", "microbiome"),
    ("bacteria", "located in", "animal"),
    ("animal", "used by", "biomedical technique"),
    ("chemical", "impact", "bacteria"),
    ("chemical", "located in", "animal"),
    ("food", "impact", "bacteria"),
    ("microbiome", "compared to", "microbiome"),
    ("human", "used by", "biomedical technique"),
    ("bacteria", "change expression", "gene"),
    ("chemical", "located in", "human"),
    ("drug", "interact", "chemical"),
    ("food", "administered", "human"),
    ("DDF", "change abundance", "bacteria"),
    ("chemical", "interact", "chemical"),
    ("chemical", "part of", "chemical"),
    ("dietary supplement", "impact", "bacteria"),
    ("DDF", "interact", "chemical"),
    ("food", "impact", "microbiome"),
    ("food", "influence", "DDF"),
    ("bacteria", "located in", "human"),
    ("dietary supplement", "administered", "human"),
    ("bacteria", "interact", "chemical"),
    ("drug", "change expression", "gene"),
    ("drug", "impact", "bacteria"),
    ("drug", "administered", "human"),
    ("anatomical location", "located in", "human"),
    ("dietary supplement", "change expression", "gene"),
    ("chemical", "change expression", "gene"),
    ("bacteria", "interact", "bacteria"),
    ("drug", "interact", "drug"),
    ("microbiome", "change expression", "gene"),
    ("bacteria", "interact", "drug"),
    ("food", "change expression", "gene")
]

# Create lookup structures for legal relations
# Map (subject_label, object_label) -> set of predicates
legal_pairs = {}
for s, p, o in LEGAL_RELATIONS:
    s = norm_ent(s); o = norm_ent(o)
    legal_pairs.setdefault((s, o), set()).add(p)

print(f"\nTotal legal relation patterns: {len(LEGAL_RELATIONS)}")
print(f"Total unique entity type pairs: {len(legal_pairs)}")

# Configuration
model_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"  # BioBERT for biomedical text
output_model_dir = "../models/bert_biomedbert_re_A5_hardneg"
max_length = 512
NEGATIVE_SAMPLE_MULTIPLIER = 5  # Number of negative samples per positive sample

print(f"\nModel: {model_name}")
print(f"Output directory: {output_model_dir}")
print(f"Negative sample multiplier: {NEGATIVE_SAMPLE_MULTIPLIER}")

Total relation labels: 18
Labels: ['no relation', 'administered', 'affect', 'change abundance', 'change effect', 'change expression', 'compared to', 'impact', 'influence', 'interact', 'is a', 'is linked to', 'located in', 'part of', 'produced by', 'strike', 'target', 'used by']

Total legal relation patterns: 55
Total unique entity type pairs: 52

Model: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Output directory: models/bert_biomedbert_re_A5_hardneg
Negative sample multiplier: 5


### BERT Model with Entity Markers
This cell implements a custom PyTorch model built on top of BERT.
The approach uses **entity marker tokens** to highlight the subject and object entities inside the input sentence.

This allows the model to focus specifically on the two entities involved in the candidate relation.

The model works as follows:

1. The input sentence contains special tokens marking the entities:
   - `[E1] ... [/E1]` for the subject
   - `[E2] ... [/E2]` for the object

2. BERT processes the sentence and produces contextual embeddings.

3. The hidden representations corresponding to the entity markers are extracted.

4. These vectors are concatenated and passed through a classification layer to predict the relation type.

In [4]:
def entity_average(hidden, mask):
    """Mean pooling over tokens belonging to the entity mention."""
    mask = mask.unsqueeze(-1).float()
    summed = (hidden * mask).sum(dim=1)
    count = mask.sum(dim=1).clamp(min=1e-6)
    return summed / count


In [5]:
class BertForREWithEntityMarkers(nn.Module):
    """
    BERT model for Relation Extraction with entity marker tokens.
    
    The model extracts hidden states at [E1] and [E2] token positions,
    concatenates them, and passes through a classification head.
    """
    
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        
        # Classification head: concatenated entity representations -> labels
        hidden_size = self.bert.config.hidden_size
        self.classifier = nn.Linear(hidden_size * 2, num_labels)
        
        self.num_labels = num_labels
    
    def forward(self, input_ids, attention_mask, e1_mask, e2_mask, labels=None):
        """
        Args:
            input_ids: Token IDs [batch_size, seq_len]
            attention_mask: Attention mask [batch_size, seq_len]
            e1_mask: Mask for [E1] token position [batch_size, seq_len]
            e2_mask: Mask for [E2] token position [batch_size, seq_len]
            labels: Ground truth labels [batch_size]
        """
        # Get BERT outputs
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        sequence_output = outputs.last_hidden_state  # [batch_size, seq_len, hidden_size]
        
        # Mention-mean pooling: media sui token della mention
        e1_h = entity_average(sequence_output, e1_mask)
        e2_h = entity_average(sequence_output, e2_mask)
        
        # Concatenate entity representations
        concat_h = torch.cat([e1_h, e2_h], dim=-1)  # [batch_size, hidden_size * 2]
        concat_h = self.dropout(concat_h)
        
        # Classification
        logits = self.classifier(concat_h)  # [batch_size, num_labels]
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
        
        return {
            'loss': loss,
            'logits': logits
        }


print("BERT RE model class defined")

BERT RE model class defined


## Data Loading Functions

In [6]:
def load_re_data(file_paths):
    """Load relation extraction data from multiple JSON files."""
    all_data = {}
    
    for file_path in file_paths:
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            all_data.update(data)
            print(f"Loaded {len(data)} documents from {os.path.basename(file_path)}")
        else:
            print(f"Warning: {file_path} not found")
    
    return all_data


print("Data loading function defined")

Data loading function defined


## Load Training and Dev Data

In [7]:
# Load training data from three quality levels
train_files = [
    "../../../data/GutBrainIE_Full_Collection_2026/Annotations/Train/gold_quality/json_format/train_gold.json",
    "../../../data/GutBrainIE_Full_Collection_2026/Annotations/Train/silver_quality/json_format/train_silver.json",
    "../../../data/GutBrainIE_Full_Collection_2026/Annotations/Train/bronze_quality/json_format/train_bronze.json",
    # opzionale (se vuoi includerlo):
    "../../../data/GutBrainIE_Full_Collection_2026/Annotations/Train/silver_quality/json_format/train_silver_2025.json",
]

train_data = load_re_data(train_files)
print(f"\nTotal training documents: {len(train_data)}")

Loaded 639 documents from train_gold.json
Loaded 811 documents from train_silver.json


Loaded 2972 documents from train_bronze.json
Loaded 499 documents from train_silver_2025.json

Total training documents: 4921


In [8]:
# Load dev data
dev_data = load_re_data([
    "../../../data/GutBrainIE_Full_Collection_2026/Annotations/Dev/json_format/dev.json"
])
print(f"Total dev documents: {len(dev_data)}")

Loaded 80 documents from dev.json
Total dev documents: 80


## Prepare Relation Extraction Examples

For each document:
1. Extract positive relation examples from annotations
2. Generate negative examples by pairing entities that are NOT related
3. Apply the negative sample multiplier to balance the dataset

In [9]:
from collections import defaultdict
def create_full_text_with_offsets(title, abstract):
    """
    Create full text by concatenating title and abstract.
    Returns full text and offset for abstract entities.
    """
    full_text = f"{title} {abstract}"
    abstract_offset = len(title) + 1
    return full_text, abstract_offset


def adjust_entity_positions(entity, abstract_offset):
    """
    Adjust entity character positions to account for title + abstract concatenation.
    """
    if entity['location'] == 'abstract':
        return {
            'start_idx': entity['start_idx'] + abstract_offset,
            'end_idx': entity['end_idx'] + abstract_offset,
            'text_span': entity['text_span'],
            'label': entity['label']
        }
    else:
        return {
            'start_idx': entity['start_idx'],
            'end_idx': entity['end_idx'],
            'text_span': entity['text_span'],
            'label': entity['label']
        }


MAX_PAIR_CHARS = 400  # prova 300/400/500

def char_distance(a, b):
    # distanza tra due mention (start inclusive)
    return abs(a["start_idx"] - b["start_idx"])
def prepare_re_examples(data, negative_multiplier=1, legal_pairs=None):
    """
    Prepare relation extraction examples with positive and negative samples.
    Only considers entity pairs that match legal relation patterns.

    Args:
        data: Dictionary of documents with entities and mention_level_relations
        negative_multiplier: Number of negative samples per positive sample
        legal_pairs: Dict mapping (subject_label, object_label) -> set(predicates)

    Returns:
        List of examples: {text, subject, object, predicate, pmid}
    """
    def loc_rank(loc: str) -> int:
        return 0 if loc == "title" else 1  # title preferred over abstract

    def best_pair(subj_cands, obj_cands):
        """
        Choose the best (subject, object) mention pair among duplicates.
        Preference:
          1) title-title > title-abstract > abstract-abstract
          2) same location preferred
          3) minimal distance in text
        """
        best = None
        best_score = None

        for s in subj_cands:
            for o in obj_cands:
                # avoid identical mention used as both
                if s["start_idx"] == o["start_idx"] and s["end_idx"] == o["end_idx"] and s["location"] == o["location"]:
                    continue

                loc_combo = loc_rank(s["location"]) + loc_rank(o["location"])
                same_loc = 0 if s["location"] == o["location"] else 1
                dist = abs(s["start_idx"] - o["start_idx"])
                score = (loc_combo, same_loc, dist)

                if best_score is None or score < best_score:
                    best_score = score
                    best = (s, o)

        return best

    examples = []

    for pmid, article in tqdm(data.items(), desc="Preparing RE examples"):
        title = article["metadata"]["title"]
        abstract = article["metadata"]["abstract"]
        full_text, abstract_offset = create_full_text_with_offsets(title, abstract)

        entities = article["entities"]
        relations = article.get("mention_level_relations", [])

        # normalize + adjust offsets
        adjusted_entities = [
            {
                **adjust_entity_positions(e, abstract_offset),
                "label": norm_ent(e["label"]),
                "text_span": norm_span(e["text_span"]),
                "location": e["location"],
            }
            for e in entities
        ]

        # index for (span,label) -> list of mentions
        ent_index = defaultdict(list)
        for e in adjusted_entities:
            ent_index[(e["text_span"], e["label"])].append(e)

        # -------- positives --------
        positive_pairs = set()

        for relation in relations:
            subj_text = norm_span(relation["subject_text_span"])
            obj_text  = norm_span(relation["object_text_span"])
            subj_lab  = norm_ent(relation["subject_label"])
            obj_lab   = norm_ent(relation["object_label"])
            pred      = relation["predicate"].strip()

            if pred not in LEGAL_RELATION_LABELS:
                continue
            if subj_lab not in LEGAL_ENTITY_LABELS or obj_lab not in LEGAL_ENTITY_LABELS:
                continue

            subj_cands = ent_index.get((subj_text, subj_lab), [])
            obj_cands  = ent_index.get((obj_text, obj_lab), [])

            pair = best_pair(subj_cands, obj_cands)
            if not pair:
                continue

            subject, obj = pair

            # optional safety: keep only legal type-pairs if provided
            type_pair = (subject["label"], obj["label"])
            if legal_pairs is not None and type_pair not in legal_pairs:
                continue

            examples.append({
                "text": full_text,
                "subject": subject,
                "object": obj,
                "predicate": pred,
                "pmid": pmid,
            })

            pair_key = (subject["start_idx"], subject["end_idx"], obj["start_idx"], obj["end_idx"])
            positive_pairs.add(pair_key)

        # -------- negatives --------
        num_negatives = len(positive_pairs) * negative_multiplier
        negative_candidates = []

        if num_negatives > 0:
            for i, subj in enumerate(adjusted_entities):
                for j, obj in enumerate(adjusted_entities):
                    if i == j:
                        continue

                    type_pair = (subj["label"], obj["label"])
                    if legal_pairs is not None and type_pair not in legal_pairs:
                        continue
                    if char_distance(subj, obj) > MAX_PAIR_CHARS:
                        continue
                    pair_key = (subj["start_idx"], subj["end_idx"], obj["start_idx"], obj["end_idx"])
                    if pair_key in positive_pairs:
                        continue

                    negative_candidates.append({
                        "text": full_text,
                        "subject": subj,
                        "object": obj,
                        "predicate": "no relation",
                        "pmid": pmid,
                    })

            if negative_candidates:
                num_to_sample = min(num_negatives, len(negative_candidates))
                # Hard negative sampling: favorisce coppie con tipi simili ai positivi
                pos_types = set()
                for ex_prev in examples:
                    if ex_prev.get("pmid") == pmid and ex_prev["predicate"] != "no relation":
                        pos_types.add(ex_prev["subject"]["label"])
                        pos_types.add(ex_prev["object"]["label"])

                hard_cands = [c for c in negative_candidates
                              if c["subject"]["label"] in pos_types or c["object"]["label"] in pos_types]
                easy_cands = [c for c in negative_candidates if c not in hard_cands]

                n_hard = min(int(num_to_sample * 0.7), len(hard_cands))
                n_easy = min(num_to_sample - n_hard, len(easy_cands))
                sampled = random.sample(hard_cands, n_hard) + random.sample(easy_cands, n_easy)
                examples.extend(sampled)

    return examples


In [10]:
# Prepare training examples
print("Preparing training examples...")
train_examples = prepare_re_examples(train_data, negative_multiplier=NEGATIVE_SAMPLE_MULTIPLIER, legal_pairs=legal_pairs)

# Count positive vs negative
positive_count = sum(1 for ex in train_examples if ex['predicate'] != 'no relation')
negative_count = sum(1 for ex in train_examples if ex['predicate'] == 'no relation')

print(f"\nTraining examples prepared: {len(train_examples)}")
print(f"  Positive examples: {positive_count}")
print(f"  Negative examples: {negative_count}")
print(f"  Ratio (neg/pos): {negative_count/positive_count:.2f}")

Preparing training examples...


Preparing RE examples:   0%|                                                                                                                                                          | 0/4921 [00:00<?, ?it/s]

Preparing RE examples:   0%|▋                                                                                                                                               | 22/4921 [00:00<00:24, 201.97it/s]

Preparing RE examples:   1%|█▊                                                                                                                                              | 60/4921 [00:00<00:16, 287.17it/s]

Preparing RE examples:   2%|██▊                                                                                                                                             | 95/4921 [00:00<00:16, 297.66it/s]

...

Preparing RE examples: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊| 4915/4921 [01:20<00:00, 26.15it/s]

Preparing RE examples: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉| 4918/4921 [01:20<00:00, 26.47it/s]

Preparing RE examples: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4921/4921 [01:20<00:00, 26.11it/s]

Preparing RE examples: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4921/4921 [01:20<00:00, 61.26it/s]


Training examples prepared: 248453
  Positive examples: 53791
  Negative examples: 194662
  Ratio (neg/pos): 3.62


In [11]:
# Prepare dev examples
print("Preparing dev examples...")
dev_examples = prepare_re_examples(dev_data, negative_multiplier=NEGATIVE_SAMPLE_MULTIPLIER, legal_pairs=legal_pairs)

positive_count_dev = sum(1 for ex in dev_examples if ex['predicate'] != 'no relation')
negative_count_dev = sum(1 for ex in dev_examples if ex['predicate'] == 'no relation')

print(f"\nDev examples prepared: {len(dev_examples)}")
print(f"  Positive examples: {positive_count_dev}")
print(f"  Negative examples: {negative_count_dev}")

Preparing dev examples...


Preparing RE examples:   0%|                                                                                                                                                            | 0/80 [00:00<?, ?it/s]

Preparing RE examples:  65%|██████████████████████████████████████████████████████████████████████████████████████████████▉                                                   | 52/80 [00:00<00:00, 450.05it/s]

Preparing RE examples: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 80/80 [00:00<00:00, 544.46it/s]


Dev examples prepared: 5061
  Positive examples: 1116
  Negative examples: 3945


In [12]:
# Show example
print("\nExample training instance:")
example = train_examples[0]
print(f"  Text: {example['text'][:150]}...")
print(f"  Subject: '{example['subject']['text_span']}' [{example['subject']['label']}]")
print(f"  Object: '{example['object']['text_span']}' [{example['object']['label']}]")
print(f"  Predicate: {example['predicate']}")


Example training instance:
  Text: Probiotics and microbial metabolites maintain barrier and neuromuscular functions and clean protein aggregation to delay disease progression in TDP43 ...
  Subject: 'α-SMA' [chemical]
  Object: 'colon' [anatomical location]
  Predicate: located in


In [13]:
print("train_docs:", len(train_data))
print("dev_docs:", len(dev_data))

print("train_examples:", len(train_examples))
print("dev_examples:", len(dev_examples))

pos = sum(1 for ex in train_examples if ex["predicate"] != "no relation")
neg = len(train_examples) - pos
print("train_pos:", pos, "train_neg:", neg, "neg/pos:", neg/max(pos,1))

train_docs: 4921
dev_docs: 80
train_examples: 248453
dev_examples: 5061
train_pos: 53791 train_neg: 194662 neg/pos: 3.6188581733003664


## Initialize Tokenizer and Add Special Tokens

In [14]:
# Initialize tokenizer
print("Initializing tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# Add special entity marker tokens
special_tokens = {"additional_special_tokens": ["[E1]", "[/E1]", "[E2]", "[/E2]"]}
tokenizer.add_special_tokens(special_tokens)

# Get token IDs for entity markers
e1_token_id = tokenizer.convert_tokens_to_ids("[E1]")
e2_token_id = tokenizer.convert_tokens_to_ids("[E2]")

print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"  Vocabulary size (with special tokens): {len(tokenizer)}")
print(f"  [E1] token ID: {e1_token_id}")
print(f"  [E2] token ID: {e2_token_id}")

Initializing tokenizer...


Tokenizer loaded: BertTokenizer
  Vocabulary size (with special tokens): 30526
  [E1] token ID: 30522
  [E2] token ID: 30524


## Tokenization with Entity Markers

In [15]:
def insert_entity_markers(text, subject, obj):
    """
    Insert entity marker tokens around subject and object entities.
    
    Args:
        text: Full text
        subject: Subject entity dict with start_idx, end_idx
        obj: Object entity dict with start_idx, end_idx
    
    Returns:
        Text with markers inserted
    """
    # Sort entities by position to insert markers correctly
    entities = [(subject['start_idx'], subject['end_idx'], '[E1]', '[/E1]'),
                (obj['start_idx'], obj['end_idx'], '[E2]', '[/E2]')]
    entities = sorted(entities, key=lambda x: x[0])
    
    # Insert markers from right to left to maintain positions
    marked_text = text
    offset = 0
    
    for start, end, start_marker, end_marker in entities:
        # Adjust positions with offset
        adj_start = start + offset
        adj_end = end + offset + 1  # +1 because end_idx is inclusive
        
        # Insert markers
        marked_text = (marked_text[:adj_start] + start_marker + 
                      marked_text[adj_start:adj_end] + end_marker + 
                      marked_text[adj_end:])
        
        # Update offset
        offset += len(start_marker) + len(end_marker)
    
    return marked_text


import re

def build_window_around_entities(text, subject, obj, window_chars=300):
    """
    Build a substring window around subject+object to avoid truncation.
    Recomputes subject/object offsets within the window.
    Assumes start/end are inclusive in the original text.
    """
    s_start, s_end = subject["start_idx"], subject["end_idx"]
    o_start, o_end = obj["start_idx"], obj["end_idx"]

    left = min(s_start, o_start)
    right = max(s_end, o_end)

    # expand window
    win_start = max(0, left - window_chars)
    win_end = min(len(text) - 1, right + window_chars)  # inclusive

    window_text = text[win_start:win_end + 1]

    # shift entity indices into window coordinates
    subj_w = dict(subject)
    obj_w = dict(obj)

    subj_w["start_idx"] = s_start - win_start
    subj_w["end_idx"] = s_end - win_start
    obj_w["start_idx"] = o_start - win_start
    obj_w["end_idx"] = o_end - win_start

    # safety clamp
    for ent in (subj_w, obj_w):
        ent["start_idx"] = max(0, min(ent["start_idx"], len(window_text) - 1))
        ent["end_idx"] = max(0, min(ent["end_idx"], len(window_text) - 1))

    return window_text, subj_w, obj_w


def tokenize_re_example(
    example,
    tokenizer,
    e1_token_id,
    e2_token_id,
    max_length=512,
    window_chars=300,
    fallback_to_fulltext=True,
):
    """
    Tokenize RE example using a window around entities so marker tokens are not truncated.
    """
    # 1) window around entities (recommended)
    w_text, w_subj, w_obj = build_window_around_entities(
        example["text"], example["subject"], example["object"], window_chars=window_chars
    )
    marked_text = insert_entity_markers(w_text, w_subj, w_obj)

    # 2) tokenize
    encoding = tokenizer(
        marked_text,
        truncation=True,
        max_length=max_length,
        padding=False,
        return_tensors="pt",
    )


    input_ids = encoding["input_ids"].squeeze(0)
    attention_mask = encoding["attention_mask"].squeeze(0)

    e1_mask = (input_ids == e1_token_id).long()
    e2_mask = (input_ids == e2_token_id).long()

    # 3) if markers got lost (rare), optionally fallback to full text,
    #    otherwise raise/skip upstream
    if (e1_mask.sum().item() != 1 or e2_mask.sum().item() != 1) and fallback_to_fulltext:
        marked_text = insert_entity_markers(example["text"], example["subject"], example["object"])
        if marked_text.count("[E1]") != 1 or marked_text.count("[E2]") != 1:
            return None
        encoding = tokenizer(
            marked_text,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors="pt",
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        e1_mask = (input_ids == e1_token_id).long()
        e2_mask = (input_ids == e2_token_id).long()

    # 4) final check: if markers missing, skip example
    if e1_mask.sum().item() != 1 or e2_mask.sum().item() != 1:
        if example.get("pmid") == "38606018" or example.get("pmid") == 38606018:
            print("BAD MARKERS DEBUG pmid", example.get("pmid"))
            print("subj", example["subject"]["text_span"], example["subject"]["start_idx"], example["subject"]["end_idx"])
            print("obj ", example["object"]["text_span"], example["object"]["start_idx"], example["object"]["end_idx"])
        return None


    label = label2id[example["predicate"]]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "e1_mask": e1_mask,
        "e2_mask": e2_mask,
        "labels": torch.tensor(label, dtype=torch.long),
    }


print("Tokenization functions defined")

Tokenization functions defined


In [16]:
# Test tokenization
test_example = train_examples[0]
tokenized = tokenize_re_example(test_example, tokenizer, e1_token_id, e2_token_id)

print("Test tokenization:")
print(f"  Input IDs shape: {tokenized['input_ids'].shape}")
print(f"  E1 mask sum (should be 1): {tokenized['e1_mask'].sum().item()}")
print(f"  E2 mask sum (should be 1): {tokenized['e2_mask'].sum().item()}")
print(f"  Label: {tokenized['labels'].item()} ({id2label[tokenized['labels'].item()]})")

# Show marked text
marked = insert_entity_markers(test_example['text'], test_example['subject'], test_example['object'])
print(f"\nMarked text preview: {marked[:200]}...")

Test tokenization:
  Input IDs shape: torch.Size([146])
  E1 mask sum (should be 1): 1
  E2 mask sum (should be 1): 1
  Label: 12 (located in)

Marked text preview: Probiotics and microbial metabolites maintain barrier and neuromuscular functions and clean protein aggregation to delay disease progression in TDP43 mutation mice. Amyotrophic lateral sclerosis (ALS)...


## Create Dataset Class
### Pre-tokenize once + tensor-only dataset (with disk cache)

In [17]:
import torch
from dataclasses import dataclass
from transformers import PreTrainedTokenizerBase

@dataclass
class REDataCollatorWithPadding:
    tokenizer: PreTrainedTokenizerBase
    pad_to_multiple_of: int | None = None

    def __call__(self, features):
        # features: list of dict {input_ids, attention_mask, e1_mask, e2_mask, labels}
        labels = torch.stack([f["labels"] for f in features])

        # usa tokenizer.pad per input_ids + attention_mask
        batch = self.tokenizer.pad(
            [{"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]} for f in features],
            padding=True,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        # pad manuale per e1/e2_mask alla stessa lunghezza del batch["input_ids"]
        max_len = batch["input_ids"].shape[1]

        def pad_1d(x, pad_value=0):
            # x: tensor [seq_len]
            if x.shape[0] == max_len:
                return x
            out = torch.full((max_len,), pad_value, dtype=x.dtype)
            out[: x.shape[0]] = x
            return out

        e1 = torch.stack([pad_1d(f["e1_mask"]) for f in features])
        e2 = torch.stack([pad_1d(f["e2_mask"]) for f in features])

        batch["e1_mask"] = e1
        batch["e2_mask"] = e2
        batch["labels"] = labels
        return batch


In [18]:
WINDOW_CHARS = 300
# ---- CONFIG CACHE PATHS ----
CACHE_DIR = os.path.join(output_model_dir, "cache_tok")
os.makedirs(CACHE_DIR, exist_ok=True)

TRAIN_CACHE = os.path.join(CACHE_DIR, f"train_dyn_maxlen{max_length}_win{WINDOW_CHARS}.pt")
DEV_CACHE   = os.path.join(CACHE_DIR, f"dev_dyn_maxlen{max_length}_win{WINDOW_CHARS}.pt")

class TensorREDataset(Dataset):
    """Custom dataset for Relation Extraction."""
    
    def __init__(self, tensor_dict):
        self.td = tensor_dict
        self.n = self.td["input_ids"].shape[0]

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        return {
            "input_ids": self.td["input_ids"][idx],
            "attention_mask": self.td["attention_mask"][idx],
            "e1_mask": self.td["e1_mask"][idx],
            "e2_mask": self.td["e2_mask"][idx],
            "labels": self.td["labels"][idx],
        }

from torch.utils.data import Dataset

class ListREDataset(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        return self.items[idx]

print("Dataset class defined")

def pretokenize_examples_dynamic(
    examples,
    tokenizer,
    e1_token_id,
    e2_token_id,
    max_length=512,
    window_chars=300,
    cache_path=None,
    verbose_every=5000,
):
    if cache_path is not None and os.path.exists(cache_path):
        print(f"[cache] Loading dynamic tokenized dataset from: {cache_path}")
        payload = torch.load(cache_path, map_location="cpu")
        return payload["items"], payload.get("skipped", [])

    print("[cache] Building dynamic tokenized items... (runs once)")
    items = []
    skipped = []

    for i, ex in enumerate(tqdm(examples, desc="Pre-tokenizing(dyn)", total=len(examples))):
        out = None
        try:
            out = tokenize_re_example(
                ex,
                tokenizer,
                e1_token_id,
                e2_token_id,
                max_length=max_length,
                window_chars=window_chars,
                fallback_to_fulltext=True,
            )
        except Exception as e:
            skipped.append((ex.get("pmid"), f"exception:{type(e).__name__}:{str(e)[:120]}"))
            continue

        if out is None:
            skipped.append((ex.get("pmid"), "tokenize_returned_None"))
            continue

        # safety: markers must exist
        if out["e1_mask"].sum().item() != 1 or out["e2_mask"].sum().item() != 1:
            skipped.append((ex.get("pmid"), f"bad_markers_e1={out['e1_mask'].sum().item()}_e2={out['e2_mask'].sum().item()}"))
            continue

        # ✅ IMPORTANT: out now contains variable-length tensors
        items.append({
            "input_ids": out["input_ids"].to(torch.int64),
            "attention_mask": out["attention_mask"].to(torch.int64),
            "e1_mask": out["e1_mask"].to(torch.int64),
            "e2_mask": out["e2_mask"].to(torch.int64),
            "labels": out["labels"].to(torch.int64),
        })

        if verbose_every and (i + 1) % verbose_every == 0:
            print(f"  ...processed {i+1}/{len(examples)} | kept={len(items)} | skipped={len(skipped)}")

    print(f"[cache] Done. kept={len(items)} / {len(examples)} | skipped={len(skipped)}")

    if cache_path is not None:
        torch.save({"items": items, "skipped": skipped}, cache_path)
        print(f"[cache] Saved dynamic tokenized dataset to: {cache_path}")

        skip_txt = cache_path.replace(".pt", "_skipped.txt")
        with open(skip_txt, "w", encoding="utf-8") as f:
            for pmid, reason in skipped:
                f.write(f"{pmid}\t{reason}\n")
        print(f"[cache] Saved skipped list to: {skip_txt}")

    return items, skipped


Dataset class defined


In [19]:
WINDOW_CHARS=300

train_items, train_skipped = pretokenize_examples_dynamic(
    train_examples, tokenizer, e1_token_id, e2_token_id,
    max_length=max_length, window_chars=WINDOW_CHARS,
    cache_path=TRAIN_CACHE
)

dev_items, dev_skipped = pretokenize_examples_dynamic(
    dev_examples, tokenizer, e1_token_id, e2_token_id,
    max_length=max_length, window_chars=WINDOW_CHARS,
    cache_path=DEV_CACHE
)

train_dataset = ListREDataset(train_items)
dev_dataset   = ListREDataset(dev_items)

collator = REDataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)


print("FAST datasets ready!")


[cache] Building dynamic tokenized items... (runs once)


Pre-tokenizing(dyn):   0%|                                                                                                                                                          | 0/248453 [00:00<?, ?it/s]

Pre-tokenizing(dyn):   0%|▏                                                                                                                                             | 223/248453 [00:00<01:53, 2194.02it/s]

Pre-tokenizing(dyn):   0%|▎                                                                                                                                             | 466/248453 [00:00<01:46, 2329.93it/s]

Pre-tokenizing(dyn):   0%|▍                                                                                                                                             | 722/248453 [00:00<01:41, 2433.47it/s]

...

Pre-tokenizing(dyn):  91%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌            | 4623/5061 [00:01<00:00, 2480.70it/s]

Pre-tokenizing(dyn):  96%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉     | 4883/5061 [00:02<00:00, 2489.36it/s]

Pre-tokenizing(dyn): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5061/5061 [00:02<00:00, 2405.82it/s]

  ...processed 5000/5061 | kept=5000 | skipped=0
[cache] Done. kept=5061 / 5061 | skipped=0


[cache] Saved dynamic tokenized dataset to: models/bert_biomedbert_re_A5_hardneg\cache_tok\dev_dyn_maxlen512_win300.pt
[cache] Saved skipped list to: models/bert_biomedbert_re_A5_hardneg\cache_tok\dev_dyn_maxlen512_win300_skipped.txt
FAST datasets ready!


## Initialize Model

In [20]:
# Initialize model
print("Initializing BERT RE model...")
model = BertForREWithEntityMarkers(model_name, num_labels=len(RELATION_LABELS))

# Resize token embeddings to account for new special tokens
model.bert.resize_token_embeddings(len(tokenizer))

print(f"Model initialized")
print(f"  Number of labels: {model.num_labels}")
print(f"  Hidden size: {model.bert.config.hidden_size}")

Initializing BERT RE model...


Loading weights:   0%|                                                                                                                                                                 | 0/199 [00:00<?, ?it/s]

Loading weights:   1%|▌                                                                                                                 | 1/199 [00:00<?, ?it/s, Materializing param=embeddings.LayerNorm.bias]

Loading weights:   1%|▌                                                                                                                 | 1/199 [00:00<?, ?it/s, Materializing param=embeddings.LayerNorm.bias]

Loading weights:   1%|█                                                                                                     | 2/199 [00:00<00:00, 1871.62it/s, Materializing param=embeddings.LayerNorm.weight]

...

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 1183.61it/s, Materializing param=pooler.dense.weight]

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 1199.01it/s, Materializing param=pooler.dense.weight]


BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | Details
-------------------------------------------+------------+--------
cls.seq_relationship.weight                | UNEXPECTED |        
cls.predictions.decoder.bias               | UNEXPECTED |        
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |        
cls.predictions.bias                       | UNEXPECTED |        
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |        
cls.predictions.transform.dense.weight     | UNEXPECTED |        
cls.seq_relationship.bias                  | UNEXPECTED |        
cls.predictions.decoder.weight             | UNEXPECTED |        
cls.predictions.transform.dense.bias       | UNEXPECTED |        

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model initialized
  Number of labels: 18
  Hidden size: 768


## Custom Trainer for Entity Marker Model

In [21]:
import numpy as np
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    pos_label_ids = [idx for label, idx in label2id.items() if label != "no relation"]
    macro_f1 = f1_score(labels, preds, labels=pos_label_ids, average="macro", zero_division=0)
    micro_f1 = f1_score(labels, preds, labels=pos_label_ids, average="micro", zero_division=0)
    return {"macro_f1_pos": macro_f1, "micro_f1_pos": micro_f1}

print("compute_metrics defined")


compute_metrics defined


In [22]:
import os
import torch
from transformers import Trainer

class RETrainer(Trainer):
    """Custom Trainer that handles entity marker masks + safe saving on Windows."""

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs, labels=labels)
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss

    # ---- CRITICAL PATCH: override _save to avoid safetensors ----
    def _save(self, output_dir: str, state_dict=None):
        os.makedirs(output_dir, exist_ok=True)

        if state_dict is None:
            state_dict = self.model.state_dict()

        # make tensors contiguous (extra-safe)
        for k, v in state_dict.items():
            if isinstance(v, torch.Tensor) and not v.is_contiguous():
                state_dict[k] = v.contiguous()

        # save as classic pytorch bin
        torch.save(state_dict, os.path.join(output_dir, "pytorch_model.bin"))

        # (optional but nice) also save training args
        torch.save(self.args, os.path.join(output_dir, "training_args.bin"))

## Configure Training Arguments

In [23]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=output_model_dir,

    learning_rate=2e-5,
    warmup_ratio=0.06,
    lr_scheduler_type="linear",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    max_grad_norm=1.0,

    num_train_epochs=3,
    weight_decay=0.01,

    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1_pos",
    greater_is_better=True,
    disable_tqdm=False,
    logging_steps=50,

    fp16=False,
    bf16=torch.cuda.is_available(),    # RTX 5070: bf16 obbligatorio
    tf32=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,

    seed=SEED,
    report_to="none"
)

print("Training configuration ready")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Learning rate: {training_args.learning_rate}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training configuration ready
  Batch size: 16
  Epochs: 3
  Learning rate: 2e-05


## Train Model

**Note:** This cell might take several minutes to hours depending on dataset size and hardware.

**Hyperparameters to experiment with:**
- `NEGATIVE_SAMPLE_MULTIPLIER`: Try 1, 2, 3, 5
- `learning_rate`: Try 1e-5, 2e-5, 3e-5
- `num_train_epochs`: Try 3, 5, 10
- `per_device_train_batch_size`: Adjust based on GPU memory
- Different pretrained models: "allenai/scibert_scivocab_uncased", "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"

In [24]:
# Initialize trainer
print("Initializing Trainer...")
trainer = RETrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

print("Trainer initialized")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Evaluation samples: {len(dev_dataset)}")

Initializing Trainer...
Trainer initialized
  Training samples: 248421
  Evaluation samples: 5061


In [25]:
# Start training
print("="*60)
print("Starting model training...")
print("="*60)

import time
training_start_time = time.time()

train_result = trainer.train()

training_duration = time.time() - training_start_time

print("\n" + "="*60)
print("TRAINING COMPLETED!")
print("="*60)
print(f"Training time: {training_duration/60:.2f} minutes")

Starting model training...


Step,Training Loss,Validation Loss,Macro F1 Pos,Micro F1 Pos
2000,0.328307,0.432839,0.501413,0.639398
4000,0.232530,0.394211,0.533547,0.659274
6000,0.224479,0.377154,0.599264,0.673621
8000,0.222926,0.374548,0.586930,0.689423
10000,0.228802,0.357804,0.603656,0.698630
12000,0.177419,0.368143,0.594914,0.688889
14000,0.163849,0.377733,0.639802,0.695446
16000,0.117076,0.410309,0.627757,0.695735
18000,0.140973,0.451712,0.631213,0.697583
20000,0.115762,0.444126,0.621851,0.694789



TRAINING COMPLETED!
Training time: 79.51 minutes


## Save Trained Model

In [26]:
# Save the trained model
print("Saving trained model...")

os.makedirs(output_model_dir, exist_ok=True)
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)

# Save label mappings
with open(os.path.join(output_model_dir, 'label_mappings.json'), 'w') as f:
    json.dump({'label2id': label2id, 'id2label': id2label}, f, indent=2)

print(f"Model saved to: {output_model_dir}")

Saving trained model...


Model saved to: models/bert_biomedbert_re_A5_hardneg
